In [10]:
from stator_pipeline import StatorConfig, generate_single
from stator_pipeline.visualiser import StatorVisualiser

# ── 1. Configure a 3-D laminated stator ──────────────────────────────────────
cfg = StatorConfig(
    # Cross-section geometry
    R_outer         = 0.25,
    R_inner         = 0.15,
    n_slots         = 36,
    slot_shape      = "SEMI_CLOSED",
    winding_type    = "DOUBLE_LAYER",
    slot_depth      = 0.06,
    slot_width_outer= 0.012,
    slot_width_inner= 0.010,
    slot_opening    = 0.004,
    slot_opening_depth = 0.003,

    # Lamination stack (triggers 3-D extrusion)
    n_lam    = 200,         # 200 sheets → stack_length = 0.07 m
    t_lam    = 0.00035,     # 0.35 mm per lamination
    z_spacing = 0.0,        # pressed flush (no inter-lamination gap)
    material = "M270_35A",

    # Mesh sizing
    mesh_yoke  = 0.006,
    mesh_slot  = 0.003,
    mesh_coil  = 0.0015,
    mesh_ins   = 0.0007,
    mesh_boundary_layers = 3,
)

# ── 2. Generate mesh (MSH + VTK + JSON) ──────────────────────────────────────
result = generate_single(
    cfg,
    output_dir = "/tmp/stator_3d",
    formats    = "MSH|VTK|JSON",
)

if not result["success"]:
    raise RuntimeError(f"Pipeline failed: {result['error']}")

print(f"MSH  : {result['msh_path']}")
print(f"VTK  : {result['vtk_path']}")
print(f"JSON : {result['json_path']}")

import json
with open(result["json_path"]) as f:
    meta = json.load(f)
stats = meta["mesh_stats"]
print(f"Nodes      : {stats['n_nodes']}")
print(f"2-D elems  : {stats['n_elements_2d']}")
print(f"3-D elems  : {stats['n_elements_3d']}")
print(f"Avg quality: {stats['avg_quality']:.3f}")

# ── 3. Visualise cross-section (matplotlib, 2-D slice) ───────────────────────
vis = StatorVisualiser()
vis.plot_cross_section(result["vtk_path"], output_png="/tmp/stator_3d_section.png")

# ── 4. Visualise full 3-D volume (pyvista) ───────────────────────────────────
try:
    import pyvista as pv

    grid = pv.read(result["vtk_path"])

    # Colour cells by region type (physical group tag stored as scalar)
    plotter = pv.Plotter()
    plotter.add_mesh(
        grid,
        scalars    = grid.cell_data.keys()[0],  # first scalar = RegionType tag
        cmap       = "tab20",
        show_edges = False,
        opacity    = 0.9,
    )
    plotter.add_scalar_bar("Region tag")
    plotter.show_axes()
    plotter.show(title="Stator — 3-D lamination stack")

except ImportError:
    print("pyvista not installed; skipping interactive 3-D view.")
    print("Install with:  pip install pyvista")

ImportError: _stator_core not available; build with -DSTATOR_WITH_PYTHON=ON